In [147]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
from scipy import interpolate

In [148]:
start_date = '1990-01-01'
end_date = '2025-04-02'

# Create a date range for all months in the period
date_range = pd.date_range(start=start_date, end=end_date, freq='QS')
master_df = pd.DataFrame({'Date': date_range})

print(f"Created quarterly DataFrame with {len(master_df)} records from {master_df['Date'].min().strftime('%Y-%m-%d')} to {master_df['Date'].max().strftime('%Y-%m-%d')}")

Created quarterly DataFrame with 142 records from 1990-01-01 to 2025-04-01


In [149]:
House_Index = pd.read_csv(os.path.join('New_Data', 'DC_House_Price_Index.csv'))

# Convert the 'observation_date' column to datetime format

House_Index['observation_date'] = pd.to_datetime(House_Index['observation_date'])
# Rename observation_date to Date
House_Index.rename(columns={'observation_date': 'Date'}, inplace=True)
# Convert the 'Date' column to datetime format
House_Index['Date'] = pd.to_datetime(House_Index['Date'])
# Rename House Index Column
House_Index.rename(columns={'ATNHPIUS47894Q': 'House_Index'}, inplace=True)

# Merge the House_Index dolumn with the monthly_df DataFrame
master_df = pd.merge(master_df, House_Index[['Date', 'House_Index']], on='Date', how='left')

print(master_df.head())

        Date  House_Index
0 1990-01-01       102.11
1 1990-04-01       102.41
2 1990-07-01       101.87
3 1990-10-01       100.53
4 1991-01-01       101.07


In [150]:
cpi_df = pd.read_csv(os.path.join('New_Data', 'DMV_CPI.csv'))

# Convert from wide to long format
cpi_long = pd.melt(
    cpi_df,
    id_vars=['Year'],
    value_vars=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    var_name='Month',
    value_name='CPI'
)

# Map month names to numbers
month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 
             'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
cpi_long['Month'] = cpi_long['Month'].map(month_map)

# Create a proper Date column (always first of the month)
cpi_long['Date'] = pd.to_datetime(dict(year=cpi_long['Year'], month=cpi_long['Month'], day=1))

# Keep only Date and CPI
cpi_clean = cpi_long[['Date', 'CPI']].sort_values('Date').reset_index(drop=True)

print(cpi_clean.head())

# Merge the CPI data with the master_df DataFrame
master_df = pd.merge(master_df, cpi_clean, on='Date', how='left')

print(master_df.head())

        Date    CPI
0 1978-01-01  64.30
1 1978-02-01  64.50
2 1978-03-01  64.70
3 1978-04-01  65.25
4 1978-05-01  65.80
        Date  House_Index    CPI
0 1990-01-01       102.11  132.0
1 1990-04-01       102.41  133.9
2 1990-07-01       101.87  135.7
3 1990-10-01       100.53  138.2
4 1991-01-01       101.07  139.1


In [151]:
# Load your data
Poverty_Rate = pd.read_csv(os.path.join('New_Data', 'Poverty_Rate.csv'))

# Convert Year to datetime (Jan 1st of each year)
Poverty_Rate['Date'] = pd.to_datetime(Poverty_Rate['Year'].astype(str) + '-01-01')
Poverty_Rate = Poverty_Rate[['Date', 'Poverty Rate (%)']].sort_values('Date')

# Set Date as index
Poverty_Rate.set_index('Date', inplace=True)

# Resample to quarterly start and interpolate
quarterly_df = Poverty_Rate.resample('QS').interpolate(method='linear')

# Reset index so Date is a column again
quarterly_df.reset_index(inplace=True)

print(quarterly_df.head(10))

quarterly_df.rename(columns={'Poverty Rate (%)': 'Poverty_Rate'}, inplace=True)

# Merge the Poverty Rate data with the master_df DataFrame
master_df = pd.merge(master_df, quarterly_df, on='Date', how='left')

print(master_df.head(60))

        Date  Poverty Rate (%)
0 1990-01-01            13.500
1 1990-04-01            13.675
2 1990-07-01            13.850
3 1990-10-01            14.025
4 1991-01-01            14.200
5 1991-04-01            14.350
6 1991-07-01            14.500
7 1991-10-01            14.650
8 1992-01-01            14.800
9 1992-04-01            14.875
         Date  House_Index     CPI  Poverty_Rate
0  1990-01-01       102.11  132.00        13.500
1  1990-04-01       102.41  133.90        13.675
2  1990-07-01       101.87  135.70        13.850
3  1990-10-01       100.53  138.20        14.025
4  1991-01-01       101.07  139.10        14.200
5  1991-04-01       101.22  140.10        14.350
6  1991-07-01       100.41  140.90        14.500
7  1991-10-01       102.01  142.95        14.650
8  1992-01-01       102.32  142.90        14.800
9  1992-04-01       101.50  143.10        14.875
10 1992-07-01       102.32  144.80        14.950
11 1992-10-01       102.43  146.45        15.025
12 1993-01-01       10

In [152]:
Unemployment_Rate = pd.read_csv(os.path.join('New_Data', 'DC_Unemployment_Rate.csv'))

# Convert date to datetime
Unemployment_Rate['Label'] = pd.to_datetime(Unemployment_Rate['Label'])
# Rename Label to Date
Unemployment_Rate.rename(columns={'Label': 'Date'}, inplace=True)
# Convert Unemployment Rate to float
Unemployment_Rate['Value'] = Unemployment_Rate['Value'].astype(float)
# Rename Value column to Unemployment_Rate
Unemployment_Rate.rename(columns={'Value': 'Unemployment_Rate'}, inplace=True)

print(Unemployment_Rate.head())

# We're going to make a reasonable assumption with economic trends, and add in a data point on 04/01/2025, and it's going to be the same as February, 4.5
added_row = pd.DataFrame({'Date': [pd.to_datetime('2025-04-01')], 'Unemployment_Rate': [4.5]})
Unemployment_Rate = pd.concat([Unemployment_Rate, added_row], ignore_index=True)

# Merge the Unemployment Rate data with the master_df DataFrame
master_df = pd.merge(master_df, Unemployment_Rate, on='Date', how='left')
print(master_df.head(40))




        Date  Unemployment_Rate
0 1990-01-01                3.8
1 1990-02-01                3.9
2 1990-03-01                3.8
3 1990-04-01                3.8
4 1990-05-01                4.1
         Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate
0  1990-01-01       102.11  132.00        13.500                3.8
1  1990-04-01       102.41  133.90        13.675                3.8
2  1990-07-01       101.87  135.70        13.850                4.7
3  1990-10-01       100.53  138.20        14.025                4.9
4  1991-01-01       101.07  139.10        14.200                5.3
5  1991-04-01       101.22  140.10        14.350                5.2
6  1991-07-01       100.41  140.90        14.500                6.1
7  1991-10-01       102.01  142.95        14.650                6.2
8  1992-01-01       102.32  142.90        14.800                6.6
9  1992-04-01       101.50  143.10        14.875                6.3
10 1992-07-01       102.32  144.80        14.950            

In [153]:
Median_Household_Income = pd.read_csv(os.path.join('New_Data', 'DC_Median_Household_Income_Annually.csv'))

# Convert observation_date to datetime
Median_Household_Income['observation_date'] = pd.to_datetime(Median_Household_Income['observation_date'])
# Rename observation_date to Date
Median_Household_Income.rename(columns={'observation_date': 'Date'}, inplace=True)

# Set date as index
Median_Household_Income.set_index('Date', inplace=True)

# Resample to quarterly start and interpolate
quarterly_income_df = Median_Household_Income.resample('QS').interpolate(method='linear')

# Reset index so Date is a column again
quarterly_income_df.reset_index(inplace=True)

print(quarterly_income_df.head(10))

# Rename the column to Median_Household_Income
quarterly_income_df.rename(columns={'MEHOINUSDCA672N': 'Median_Household_Income'}, inplace=True)

# Fill in the missing values until 10/01/2024 using linear interpolation
quarterly_income_df['Median_Household_Income'] = quarterly_income_df['Median_Household_Income'].interpolate(method='linear')

# Merge the Median Household Income data with the master_df DataFrame
master_df = pd.merge(master_df, quarterly_income_df, on='Date', how='left')

print(master_df.tail())


        Date  MEHOINUSDCA672N
0 1984-01-01          53650.0
1 1984-04-01          53635.0
2 1984-07-01          53620.0
3 1984-10-01          53605.0
4 1985-01-01          53590.0
5 1985-04-01          55402.5
6 1985-07-01          57215.0
7 1985-10-01          59027.5
8 1986-01-01          60840.0
9 1986-04-01          62237.5
          Date  House_Index       CPI  Poverty_Rate  Unemployment_Rate  \
137 2024-04-01       392.17  314.3875          11.1                3.4   
138 2024-07-01       394.52  316.4450          11.1                4.6   
139 2024-10-01       392.54  317.0635          11.1                4.1   
140 2025-01-01          NaN  318.1750          11.1                4.3   
141 2025-04-01          NaN       NaN           NaN                4.5   

     Median_Household_Income  
137                      NaN  
138                      NaN  
139                      NaN  
140                      NaN  
141                      NaN  


In [154]:
# Load the DC Population data
Population = pd.read_csv(os.path.join('New_Data', 'DC_Population.csv'))

# Convert observation_date to datetime
Population['observation_date'] = pd.to_datetime(Population['observation_date'])

# Rename WSHPOP to Population
Population.rename(columns={'observation_date': 'Date', 'WSHPOP': 'Population'}, inplace=True)

# Set date as index
Population.set_index('Date', inplace=True)

# Resample to quarterly start and interpolate
quarterly_population_df = Population.resample('QS').interpolate(method='linear')

# Reset index so Date is a column again
quarterly_population_df.reset_index(inplace=True)

print(quarterly_population_df.head(10))

# Create the row 1990-01-01, take the difference between the first two rows and subtract to get the population for 1990-01-01
new_row = pd.DataFrame({'Date': [pd.to_datetime('1990-01-01')], 'Population': [quarterly_population_df['Population'].iloc[0] - (quarterly_population_df['Population'].iloc[1] - quarterly_population_df['Population'].iloc[0])]})

quarterly_population_df = pd.concat([new_row, quarterly_population_df], ignore_index=True)

print(quarterly_population_df.head(10))

# Merge the Population data with the master_df DataFrame
master_df = pd.merge(master_df, quarterly_population_df, on='Date', how='left')
print(master_df.head(40))

        Date   Population
0 1990-04-01  4105.955000
1 1990-07-01  4123.395878
2 1990-10-01  4140.836756
3 1991-01-01  4158.277634
4 1991-04-01  4175.718512
5 1991-07-01  4193.159390
6 1991-10-01  4210.600268
7 1992-01-01  4228.041146
8 1992-04-01  4245.482024
9 1992-07-01  4262.922902
        Date   Population
0 1990-01-01  4088.514122
1 1990-04-01  4105.955000
2 1990-07-01  4123.395878
3 1990-10-01  4140.836756
4 1991-01-01  4158.277634
5 1991-04-01  4175.718512
6 1991-07-01  4193.159390
7 1991-10-01  4210.600268
8 1992-01-01  4228.041146
9 1992-04-01  4245.482024
         Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate  \
0  1990-01-01       102.11  132.00        13.500                3.8   
1  1990-04-01       102.41  133.90        13.675                3.8   
2  1990-07-01       101.87  135.70        13.850                4.7   
3  1990-10-01       100.53  138.20        14.025                4.9   
4  1991-01-01       101.07  139.10        14.200                5.3   
5 

In [155]:
Interest_Rate = pd.read_csv(os.path.join('New_Data', 'Interest_Rates.csv'))

# Convert observation_date to datetime
Interest_Rate['observation_date'] = pd.to_datetime(Interest_Rate['observation_date'])
# Rename observation_date to Date
Interest_Rate.rename(columns={'observation_date': 'Date'}, inplace=True)
# Convert FEDFUNDS to float
Interest_Rate['FEDFUNDS'] = Interest_Rate['FEDFUNDS'].astype(float)
# Rename FEDFUNDS column to Interest_Rate
Interest_Rate.rename(columns={'FEDFUNDS': 'Interest_Rate'}, inplace=True)

# Add new row for 2025-04-01
Interest_Rate.loc[len(Interest_Rate)] = [pd.Timestamp('2025-04-01'), 4.33]

# Merge the Interest Rate data with the master_df DataFrame
master_df = pd.merge(master_df, Interest_Rate, on='Date', how='left')

print(master_df.head())

        Date  House_Index    CPI  Poverty_Rate  Unemployment_Rate  \
0 1990-01-01       102.11  132.0        13.500                3.8   
1 1990-04-01       102.41  133.9        13.675                3.8   
2 1990-07-01       101.87  135.7        13.850                4.7   
3 1990-10-01       100.53  138.2        14.025                4.9   
4 1991-01-01       101.07  139.1        14.200                5.3   

   Median_Household_Income   Population  Interest_Rate  
0                  58390.0  4088.514122           8.23  
1                  59160.0  4105.955000           8.26  
2                  59930.0  4123.395878           8.15  
3                  60700.0  4140.836756           8.11  
4                  61470.0  4158.277634           6.91  


In [156]:
# Load data
Mortgage_Rate = pd.read_csv(os.path.join('New_Data', 'Mortgage_Rate.csv'))

# Convert observation_date to datetime and rename
Mortgage_Rate['observation_date'] = pd.to_datetime(Mortgage_Rate['observation_date'])
Mortgage_Rate.rename(columns={'observation_date': 'Date', 'MORTGAGE30US': 'Mortgage_Rate'}, inplace=True)

# Convert to float, if needed
Mortgage_Rate['Mortgage_Rate'] = Mortgage_Rate['Mortgage_Rate'].astype(float)

# Set Date as index to resample
Mortgage_Rate.set_index('Date', inplace=True)

# Resample to monthly start (or use 'Q' for quarterly average)
Mortgage_Rate = Mortgage_Rate.resample('MS').mean().reset_index()

# Merge with master_df on Date
master_df = pd.merge(master_df, Mortgage_Rate, on='Date', how='left')

print(master_df.head())

        Date  House_Index    CPI  Poverty_Rate  Unemployment_Rate  \
0 1990-01-01       102.11  132.0        13.500                3.8   
1 1990-04-01       102.41  133.9        13.675                3.8   
2 1990-07-01       101.87  135.7        13.850                4.7   
3 1990-10-01       100.53  138.2        14.025                4.9   
4 1991-01-01       101.07  139.1        14.200                5.3   

   Median_Household_Income   Population  Interest_Rate  Mortgage_Rate  
0                  58390.0  4088.514122           8.23         9.8950  
1                  59160.0  4105.955000           8.26        10.3700  
2                  59930.0  4123.395878           8.15        10.0350  
3                  60700.0  4140.836756           8.11        10.1775  
4                  61470.0  4158.277634           6.91         9.6375  


In [157]:
Gross_Domestic_Product = pd.read_csv(os.path.join('New_Data', 'GDP_growth.csv'))

# Convert observation_date to datetime
Gross_Domestic_Product['observation_date'] = pd.to_datetime(Gross_Domestic_Product['observation_date'])
# Rename observation_date to Date
Gross_Domestic_Product.rename(columns={'observation_date': 'Date'}, inplace=True)
# Convert GDP to float
Gross_Domestic_Product['GDP'] = Gross_Domestic_Product['GDP'].astype(float)

# Merge the GDP data with the master_df DataFrame
master_df = pd.merge(master_df, Gross_Domestic_Product, on='Date', how='left')

print(master_df.tail(20))


          Date  House_Index       CPI  Poverty_Rate  Unemployment_Rate  \
122 2020-07-01       293.29  267.2870        11.550                9.5   
123 2020-10-01       298.72  268.7440        11.575                8.5   
124 2021-01-01       304.73  270.5350        11.600                7.5   
125 2021-04-01       318.67  274.0845        11.575                7.0   
126 2021-07-01       330.13  279.0990        11.550                7.3   
127 2021-10-01       336.72  282.5865        11.525                5.4   
128 2022-01-01       345.63  286.6780        11.500                5.0   
129 2022-04-01       362.16  294.3930        11.400                3.5   
130 2022-07-01       360.67  299.9370        11.300                3.9   
131 2022-10-01       356.32  299.6765        11.200                3.5   
132 2023-01-01       357.48  299.1490        11.100                3.6   
133 2023-04-01       369.72  304.2720        11.100                2.6   
134 2023-07-01       372.73  305.2730 

In [159]:
# Ensure the index of master_df is of datetime type
if not pd.api.types.is_datetime64_any_dtype(master_df.index):
	master_df.set_index('Date', inplace=True)
	master_df.index = pd.to_datetime(master_df.index)

# 1. Create df_future with data from 2025-01-01 onwards from the original master_df
#    It's crucial to do this BEFORE interpolating the main dataframe
df_future = master_df.loc[master_df.index >= pd.Timestamp('2025-01-01')].copy()

# 2. Create df_filled with data up to 2024-10-01 for interpolation
#    This separates the data that needs filling from the future data.
df_filled = master_df.loc[master_df.index <= pd.Timestamp('2024-10-01')].copy()

# 3. Interpolate missing values in df_filled using linear method
df_filled.interpolate(method='linear', inplace=True)

# 4. Verify that there are no more missing values in df_filled
print("Missing values in df_filled after interpolation:")
print(df_filled.isnull().sum().to_markdown(numalign="left", stralign="left"))

# 5. Display the info for the filled df_filled
print("\nInfo for df_filled (data up to 2024-10-01, filled):")
print(df_filled.info())

# 6. Display the df_future DataFrame
print("\ndf_future (data for 2025-01-01 and 2025-04-01):")
print(df_future.to_markdown(numalign="left", stralign="left"))

# 7. Display the last few rows of df_filled to confirm the date range
print("\nLast 5 rows of df_filled:")
print(df_filled.tail().to_markdown(numalign="left", stralign="left"))
# --- End of Corrected Cell ---

Missing values in df_filled after interpolation:
|                         | 0   |
|:------------------------|:----|
| House_Index             | 0   |
| CPI                     | 0   |
| Poverty_Rate            | 0   |
| Unemployment_Rate       | 0   |
| Median_Household_Income | 0   |
| Population              | 0   |
| Interest_Rate           | 0   |
| Mortgage_Rate           | 0   |
| GDP                     | 0   |

Info for df_filled (data up to 2024-10-01, filled):
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 140 entries, 1990-01-01 to 2024-10-01
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   House_Index              140 non-null    float64
 1   CPI                      140 non-null    float64
 2   Poverty_Rate             140 non-null    float64
 3   Unemployment_Rate        140 non-null    float64
 4   Median_Household_Income  140 non-null    float64
 5   Population        

In [160]:
# --- Start of Corrected Last Cell ---
# Save the processed DataFrame and the future data DataFrame
output_file_filled = 'DC_Master_Data_Filled.csv' # Changed filename
output_file_future = 'DC_Future_Data.csv'

# Save df_filled (interpolated data up to 2024-10-01)
# Use reset_index() to include the 'Date' column in the CSV
df_filled.reset_index().to_csv(output_file_filled, index=False)

# Save df_future (original data for 2025 onwards)
# Use reset_index() to include the 'Date' column in the CSV
df_future.reset_index().to_csv(output_file_future, index=False)

print(f"Filled data saved to {output_file_filled}")
print(f"Future data saved to {output_file_future}")
# --- End of Corrected Last Cell ---

Filled data saved to DC_Master_Data_Filled.csv
Future data saved to DC_Future_Data.csv
